# Contract details

The same notebook as [`../contract_details.ipynb`](../contract_details.ipynb), written against
[`ib_async`](https://github.com/ib-api-reloaded/ib_async) instead of the TWS API
shape. The library is unmodified and installed as usual; `ibx.ib_async.attach`
replaces the one layer of it that expects a socket to a gateway.

What the venue knows about a contract, and how it answers a description that matches more than one.

## Connecting

`IB.connect` was written for a gateway, so it takes a host, a port and a client
id. Here it takes none of them: the credentials go to `attach`, and there is no
local process to reach.

`ib.sleep()` rather than `time.sleep()` throughout. The library's loop runs on
this thread, and a plain sleep stops it — every stream then reads as dead.

In [ ]:
import os
from dotenv import load_dotenv
from ib_async import IB, util
import ibx.ib_async

util.startLoop()
load_dotenv()

ib = ibx.ib_async.attach(
    IB(),
    username=os.environ["IB_USERNAME"],
    password=os.environ["IB_PASSWORD"],
    paper=True,
)
ib.connect()          # names no host: there is no gateway to name

print(f"connected: {ib.isConnected()}")
print(f"accounts:  {ib.managedAccounts()}")

## One contract

In [ ]:
from ib_async import Stock

details = ib.reqContractDetails(Stock("AAPL", "SMART", "USD"))
print(f"{len(details)} match")

d = details[0]
print(f"conId          {d.contract.conId}")
print(f"longName       {d.longName}")
print(f"exchange       {d.contract.exchange}")
print(f"primaryExchange{d.contract.primaryExchange:>16}")
print(f"tradingHours   {d.tradingHours[:60]}")
print(f"minTick        {d.minTick}")

## A description that matches more than one

A symbol with no exchange names a family. The venue answers with every
listing, and the caller chooses.

In [ ]:
from ib_async import Contract

vague = Contract(symbol="AAPL", secType="STK", currency="USD")
matches = ib.reqContractDetails(vague)
print(f"{len(matches)} listings\n")
for m in matches[:8]:
    c = m.contract
    print(f"{c.exchange:12} {c.primaryExchange:12} {c.currency}")

## Searching by name

In [ ]:
for m in ib.reqMatchingSymbols("apple")[:8]:
    c = m.contract
    print(f"{c.symbol:8} {c.secType:6} {c.primaryExchange:10} {c.currency}")

In [ ]:
ib.disconnect()